In [1]:
import json
import os
import sys
import hashlib
from pathlib import Path
from datetime import datetime

import wandb
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

from src.utils.get_trainer import get_trainer
from src.utils.loggers import WandbLogger
from src.utils.telegram import send_message

## Create OOF and Test pred

In [2]:
# ===== User config =====
load_dotenv(dotenv_path="../../.env")

model_name = "lgbm"
data_id = "042"

n_trial = 1

n_folds = 5
seed = 42

In [3]:
# === get params, n_folds, seed, and batch_rows ===
study_name = f"{model_name}-{data_id}"
feature_dir = Path(f"../../artifacts/features/{data_id}")

params_id = f"trl{n_trial}"
params_path = f"../../artifacts/optuna/{study_name}/{params_id}.json"

with open(feature_dir / "meta.json", "r") as f:
    meta = json.load(f)

train_paths = meta["train_paths"]
test_paths = meta["test_paths"]
level = meta["level"]

with open(params_path, "r") as f:
    manifest = json.load(f)

params = manifest["params"]
opts = manifest["opts"]

print("Manifest path: ", params_path)
print("Params:\n", params)
print("opts: ", opts)

Manifest path:  ../../artifacts/optuna/lgbm-042/trl1.json
Params:
 {'learning_rate': 0.02, 'max_depth': 11, 'num_leaves': 1156, 'min_child_samples': 14667, 'min_split_gain': 0.039079671568228794, 'feature_fraction': 0.40921304830970556, 'bagging_fraction': 0.7045980821176709, 'bagging_freq': 1, 'lambda_l1': 1.574189004745663, 'lambda_l2': 0.040428727350273294}
opts:  {'earlys_stopping_rounds': 5, 'max_epochs': 20, 'min_epochs': 4}


In [4]:
# === WANDB ===
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

run = wandb.init(
    project=wandb_project,
    group=study_name,
    name=params_id,
    job_type="cv_training",
    tags=[model_name, level],
    config={
        "data_id": data_id,
        "n_folds": n_folds,
        "seed": seed,
        **params,
        **opts
    },
    dir="../../artifacts",
    reinit="finish_previous"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# === Training ===
trainer_class = get_trainer(model_name)
trainer = trainer_class(
    data_id,
    train_paths,
    test_paths,
    features=None,
    target="target",
    fold_col=None,
    weight_col=None,
    cat_cols=None,
    params=params,
    n_folds=n_folds,
    seed=seed,
    gpu=True,
    opts=opts
)

result = trainer.fit(
    loggers=[WandbLogger(run=run)]
)

Fold Col: 5fold-s42
===== Fold 1 / 5 =====
Free CPU Mem: 12.86 GB
Free GPU Mem: 7.09 GB
Training until validation scores don't improve for 500 rounds
[100]	train's auc: 0.965225	eval's auc: 0.966073
[200]	train's auc: 0.969107	eval's auc: 0.969762
[300]	train's auc: 0.970952	eval's auc: 0.971429
[400]	train's auc: 0.972274	eval's auc: 0.97256
[500]	train's auc: 0.973135	eval's auc: 0.973225
[600]	train's auc: 0.973808	eval's auc: 0.973682
[700]	train's auc: 0.974349	eval's auc: 0.974031
[800]	train's auc: 0.974775	eval's auc: 0.974265
[900]	train's auc: 0.975168	eval's auc: 0.974466
[1000]	train's auc: 0.975527	eval's auc: 0.974631
[1100]	train's auc: 0.975848	eval's auc: 0.974762
[1200]	train's auc: 0.976143	eval's auc: 0.974875
[1300]	train's auc: 0.976418	eval's auc: 0.974976
[1400]	train's auc: 0.976687	eval's auc: 0.975059
[1500]	train's auc: 0.976945	eval's auc: 0.975127
[1600]	train's auc: 0.977188	eval's auc: 0.975184
[1700]	train's auc: 0.977424	eval's auc: 0.975225
[1800]	tra

eval/f1/auc,▁▄▄▅▅▆▇▇▇▇▇▇▇███████████████████████████
eval/f2/auc,▁▁▃▄▆▆▇▇▇▇██████████████████████████████
eval/f3/auc,▁▂▂▄▅▅▇▇▇▇▇▇▇▇▇▇████████████████████████
eval/f4/auc,▁▃▅▇▇▇██████████████████████████████████
eval/f5/auc,▁▃▄▄▄▆▆▇▇▇▇▇▇███████████████████████████
iter_f1,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇████
iter_f2,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
iter_f3,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
iter_f4,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇█
iter_f5,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇███
train/f1/auc,▁▃▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████


In [6]:
# ===== Save Manifest, OOF, and Test pred=====
runs_root = Path("../../runs")
run_id = f"{model_name}-{data_id}-{params_id}-{n_folds}fold-s{seed}"
run_dir = runs_root / run_id
run_dir.mkdir(parents=True, exist_ok=True)

s = json.dumps(params, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
phash = hashlib.sha256(s.encode()).hexdigest()[:10]

manifest = {
    "run_id": run_id,
    "wandb_id": run.id,
    "wandb_url": run.url,
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_name": model_name,
    "data_id": data_id,
    "n_folds": n_folds,
    "seed": seed,
    "feature_dir": str(feature_dir.resolve()),
    "phash": phash,
    "param_source": params_path,
    "params": params,
    "run_dir": str(run_dir.resolve()),
    "cv_score": result["oof_score"],
    "submission": {
        "competition": None,
        "ref": None,
        "file": None,
        "public_score": None,
        "private_score": None,
        "submitted_at": None
    }
}

with open(run_dir / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("manifest saved", run_dir / "manifest.json")

# oof and test pred
np.save(run_dir / "oof.npy", result["oof"])
np.save(run_dir / "test.npy", result["test_pred"])

print(f"oof and test preds saved successfully in:\n{run_dir}")
send_message(
    f"✅ Finished CV for {run_id}!"
    f"\nScore: {round(result['oof_score'], 5)}"
)

manifest saved ../../runs/lgbm-042-trl1-5fold-s42/manifest.json
oof and test preds saved successfully in:
../../runs/lgbm-042-trl1-5fold-s42
✅ Message sent.


## Check Feature Importance (Only XGB or LGBM)

In [7]:
# Plot feature importance and get top columns
df = result["fi_mean"].to_pandas()
df_top100 = df[:100]

fig, ax = plt.subplots(figsize=(24, 32))
sns.barplot(
    data=df_top100,
    y="Feature",
    x="mean_ratio",
    orient="h",
    palette="flare",
    hue="Feature",
    ax=ax
)
for container in ax.containers:
    labels = ax.bar_label(container)
    for label in labels:
        label.set_fontsize(20)
plt.title("Feature Importance Top 100", fontsize=32)
plt.xlabel("Importance", fontsize=28)
plt.ylabel("Feature", fontsize=28)
ax.tick_params(axis="x", labelsize=20)
ax.tick_params(axis="y", labelsize=20)
plt.tight_layout()

fig.savefig(run_dir / "feature_importance.png", dpi=200, bbox_inches="tight")
fig.savefig(run_dir / "feature_importance.svg", bbox_inches="tight")

plt.show()

print(f"FI graph saved successfully to {run_dir}")

KeyError: 'fi_mean'

In [ ]:
thresholds = [0.90, 0.95, 0.99]       # しきい値リスト

total = df["mean_ratio"].sum()
print("All cols ", len(df))

out = {}
for th in thresholds:
    limit = th * total                # このしきい値に対応する累積比率

    # 累積と K を計算（「閾値を超える要素も1つ含める」ロジック）
    cum = df["mean_ratio"].cumsum()
    k = int((cum <= limit).sum())
    if k < len(df):
        k += 1

    drop_cols = df["Feature"].iloc[k:].tolist()
    print(f"\nThreshold {th:.2f} → Drop {len(drop_cols)} cols")
    out[str(k)] = drop_cols

path = run_dir / "drop_cols.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=4)
print(f"saved: {path}")

In [64]:
auc_oof = 0.9876557
auc_mean = 0.3445333
auc_std = 0.223445
logloss_oof = 0.9876557
logloss_mean = 0.3445333
logloss_std = 0.223445
epoch_mean = 23.56

print("\nCV Results")
print("┌──────────┬────────────┬─────────────────────┐")
print(f"│ {'Metric':^8} | {'OOF':>10} | {'Mean':>8} ± {'Std':<8} |")
print("├──────────┼────────────┼─────────────────────┤")
print(f"│ {'LOGLOSS':^8} | {logloss_oof:>10.5f} | {logloss_mean:>8.5f} ± {logloss_std:<8.5f} |")
print("├──────────┼────────────┼─────────────────────┤")
print(f"│ {'AUC':^8} | {auc_oof:>10.5f} | {auc_mean:>8.5f} ± {auc_std:<8.5f} |")
print("└──────────┴────────────┴─────────────────────┘")

print(f"\n{' CV Results ':*^48}")
print("─" * 48)
print(f" {'Metric':^9}  {'OOF':>10}  {'Mean':>10} ± {'Std':<10} ")
print("-" * 48)
print(f" {'LOGLOSS':^9} "
      f" {logloss_oof:>10.5f} "
      f" {logloss_mean:>10.5f} ± {logloss_std:<10.5f} ")
print(f" {'AUC':^9} "
      f" {auc_oof:>10.5f} "
      f" {auc_mean:>10.5f} ± {auc_std:<10.5f} ")
print("─" * 48)

print(f"Avg best epoch: {epoch_mean}")
print(f"Free CPU Mem: {round(12.3455, 2)} GB")
print(f"Free GPU Mem: {round(12.3456, 2)} GB")


CV Results
┌──────────┬────────────┬─────────────────────┐
│  Metric  |        OOF |     Mean ± Std      |
├──────────┼────────────┼─────────────────────┤
│ LOGLOSS  |    0.98766 |  0.34453 ± 0.22345  |
├──────────┼────────────┼─────────────────────┤
│   AUC    |    0.98766 |  0.34453 ± 0.22345  |
└──────────┴────────────┴─────────────────────┘

****************** CV Results ******************
────────────────────────────────────────────────
  Metric           OOF        Mean ± Std        
------------------------------------------------
  LOGLOSS      0.98766     0.34453 ± 0.22345    
    AUC        0.98766     0.34453 ± 0.22345    
────────────────────────────────────────────────
Avg best epoch: 23.56
Free CPU Mem: 12.35 GB
Free GPU Mem: 12.35 GB


In [25]:
def print_cv_results_plain(metrics: dict, avg_best_epoch: float, free_cpu_gb: float, free_gpu_gb: float):
    # 幅はお好みで
    def row(metric, oof, mean, std):
        return f"│ {metric:<8} │ {oof:>10.5f} │ {mean:>8.5f} ± {std:<8.5f} │"

    line  = "┌──────────┬────────────┬─────────────────────┐"
    head  = "│ Metric   │        OOF │         Mean ± Std  │"
    sep   = "├──────────┼────────────┼─────────────────────┤"
    tail  = "└──────────┴────────────┴─────────────────────┘"

    print("\nCV Results")
    print(line); print(head); print(sep)
    for name, m in metrics.items():
        print(row(name.upper(), m["oof"], m["mean"], m["std"]))
    print(tail)
    print(f"⏱  Avg best epoch: {avg_best_epoch:.1f}")
    print(f"🖥  Free CPU Mem:  {free_cpu_gb:.2f} GB")
    print(f"🎮  Free GPU Mem:  {free_gpu_gb:.2f} GB")

In [52]:
print(len("───────────────────────────────────────────"))

43


In [19]:
metrics = {"auc": {"oof":0.9123,"mean":0.9101,"std":0.0032},"logloss":{"oof":0.3210,"mean":0.3254,"std":0.0061}}
epoch = 23.4
cpu = 23.4
gpu = 23.4
print_cv_results_plain(metrics, epoch, cpu, gpu)


CV Results
┌──────────┬────────────┬─────────────────────┐
│ Metric   │        OOF │         Mean ± Std  │
├──────────┼────────────┼─────────────────────┤
│ AUC      │    0.91230 │  0.91010 ± 0.00320  │
│ LOGLOSS  │    0.32100 │  0.32540 ± 0.00610  │
└──────────┴────────────┴─────────────────────┘
⏱  Avg best epoch: 23.4
🖥  Free CPU Mem:  23.40 GB
🎮  Free GPU Mem:  23.40 GB


In [72]:
i = 1
n_folds = 10

title = f" Fold {i + 1} / {n_folds} "
print("=" * 48)
print(f"{title:=^48}")
print("=" * 48)
i = 100
n_folds = 10000
title = f" Fold {i + 1} / {n_folds} "
print(f"{title:=^48}")

================= Fold 2 / 10 ==================
=============== Fold 101 / 10000 ===============
